# SiDroForest — prepare final contextual maps for data publication

This notebook prepares a **lean, publication-ready dataset** containing the final contextual forest-composition maps used in the revised manuscript.

It deliberately does **not** republish the raw Sentinel-2 imagery, global land-cover products, or the original RF training data. Those inputs should continue to be cited from their existing repositories.

## Product prepared here

The final map product uses the revised manuscript definition:

- pixel-level input: previously developed SiDroForest RF hard labels;
- contextual method: **hard-label spatial composition**;
- nominal neighbourhood width: **500 m**;
- dominance threshold: **0.75**;
- output classes:
  - `0` = masked / non-forest;
  - `1` = evergreen needleleaf forest;
  - `2` = summergreen needleleaf forest;
  - `3` = mixed forest.

At 10 m pixel spacing, the centred nominal 500 m neighbourhood is implemented as a **51 × 51 pixel window**.

The notebook:

1. locates the final RF hard-label maps produced by the revised analysis;
2. regenerates the final contextual maps using the same v16 methodology;
3. stages the selected manuscript-site GeoTIFFs in a clean publication folder;
4. performs quality-control checks;
5. writes dataset metadata and class definitions;
6. creates a README and checksum manifest;
7. optionally creates a ZIP archive for upload to PANGAEA/Zenodo.


In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import os
import hashlib
import shutil
import zipfile

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from scipy.ndimage import uniform_filter

print("Setup complete.")


## 1. Configuration

By default, the notebook publishes the **eight field-linked manuscript sites** rather than every experimental raster found in the analysis folder.

If you later decide that only Lake Khamra and Yakutsk should be deposited, change:

```python
PUBLISH_SITES = ["LK", "YA"]
```

The other analysis-site codes that may be present in the RF summary are reported separately and are **not silently included**.


In [ ]:
# ============================================================
# 1. CONFIGURATION — repository version
# ============================================================
REPO_ROOT = Path(os.environ.get("SIDRO_REPO_ROOT", Path.cwd())).resolve()
ANALYSIS_OUT = Path(
    os.environ.get("SIDRO_OUTPUT_ROOT", REPO_ROOT / "outputs")
).resolve()

RF_SUMMARY = ANALYSIS_OUT / "rf_probability_map_summary.csv"
EXISTING_CONTEXT_DIR = ANALYSIS_OUT / "contextual_maps"

DATASET_ROOT = Path(
    os.environ.get(
        "SIDRO_DATASET_OUTPUT_ROOT",
        REPO_ROOT / "outputs" / "CONTEXTUAL_MAPS_PUBLICATION_v1",
    )
).resolve()
MAP_DIR = DATASET_ROOT / "maps"
METADATA_DIR = DATASET_ROOT / "metadata"
MAP_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

ADOPTED_SCALE_M = 500
ADOPTED_THRESHOLD = 0.75
ADOPTED_METHOD = "hard_label"

PUBLISH_SITES = ["BI", "LK", "ML", "NY", "SU", "SW", "VI", "YA"]
SITE_NAMES = {
    "BI": "Bilibino",
    "LK": "Lake Khamra",
    "ML": "Mirny-Lensk",
    "NY": "Nyurba",
    "SU": "Suntar",
    "SW": "Suntar West",
    "VI": "Vilnuyi",
    "YA": "Yakutsk",
}

EVERGREEN = 1
SUMMERGREEN = 2
MIXED = 3

print("RF summary:", RF_SUMMARY)
print("Dataset staging folder:", DATASET_ROOT)
print("Sites requested:", PUBLISH_SITES)


In [ ]:
# ============================================================
# 2. LOAD RF HARD-LABEL SOURCE MAPS
# ============================================================

if not RF_SUMMARY.exists():
    raise FileNotFoundError(
        f"RF summary not found: {RF_SUMMARY}\n"
        "Run the final SiDroForest methodological-analysis notebook first."
    )

rf_summary = pd.read_csv(RF_SUMMARY)

required_cols = {"site", "hard_file"}
missing_cols = required_cols - set(rf_summary.columns)
if missing_cols:
    raise ValueError(f"RF summary is missing required columns: {missing_cols}")

rf_summary["site"] = rf_summary["site"].astype(str)

available_sites = sorted(rf_summary["site"].unique().tolist())
extra_sites = sorted(set(available_sites) - set(PUBLISH_SITES))
missing_sites = sorted(set(PUBLISH_SITES) - set(available_sites))

print("Available RF sites:", available_sites)
print("Requested manuscript sites:", PUBLISH_SITES)
print("Extra analysis sites NOT selected for publication:", extra_sites)

if missing_sites:
    print("WARNING — requested sites missing from RF summary:", missing_sites)

selected = rf_summary[
    rf_summary["site"].isin(PUBLISH_SITES)
].copy()

selected["hard_file_exists"] = selected["hard_file"].map(
    lambda x: Path(str(x)).exists()
)

display(
    selected[
        ["site", "hard_file", "hard_file_exists"]
    ].sort_values("site")
)

if not selected["hard_file_exists"].all():
    bad = selected.loc[
        ~selected["hard_file_exists"],
        ["site", "hard_file"]
    ]
    raise FileNotFoundError(
        "One or more selected hard-label source rasters are missing:\n"
        + bad.to_string(index=False)
    )


## 2. Exact contextual method used for the final maps

These functions reproduce the hard-label contextual formulation used in the revised analysis.

For each centred neighbourhood, the method calculates the local proportions of valid evergreen and summergreen RF labels. A pixel is assigned to evergreen or summergreen when that class reaches the dominance threshold; otherwise it is classified as mixed.

Only the **central pixel's existing forest validity** is used to determine whether an output class is assigned, so masked/non-forest pixels remain `0`.


In [ ]:
# ============================================================
# 3. CONTEXTUAL-MAPPING FUNCTIONS — SAME LOGIC AS v16
# ============================================================

def odd_window_pixels(width_m, pixel_size_m):
    n = max(1, int(round(width_m / pixel_size_m)))
    if n % 2 == 0:
        n += 1
    return n


def local_mean_valid(values, valid, window_pixels):
    values0 = np.where(valid, values, 0).astype("float32")
    valid0 = valid.astype("float32")

    num = uniform_filter(
        values0,
        size=window_pixels,
        mode="constant",
        cval=0
    )
    den = uniform_filter(
        valid0,
        size=window_pixels,
        mode="constant",
        cval=0
    )

    out = np.divide(
        num,
        den,
        out=np.full_like(num, np.nan),
        where=den > 0
    )
    return out


def hard_context(hard, valid, window_pixels):
    eg = ((hard == EVERGREEN) & valid).astype("float32")
    sg = ((hard == SUMMERGREEN) & valid).astype("float32")

    eg_share = local_mean_valid(eg, valid, window_pixels)
    sg_share = local_mean_valid(sg, valid, window_pixels)

    total = eg_share + sg_share

    eg_share = np.divide(
        eg_share,
        total,
        out=np.full_like(eg_share, np.nan),
        where=total > 0
    )
    sg_share = np.divide(
        sg_share,
        total,
        out=np.full_like(sg_share, np.nan),
        where=total > 0
    )

    return eg_share, sg_share


def categorical_from_composition(
    eg_share,
    sg_share,
    valid,
    threshold
):
    out = np.zeros(valid.shape, dtype="uint8")

    out[
        valid & (eg_share >= threshold)
    ] = EVERGREEN

    out[
        valid & (sg_share >= threshold)
    ] = SUMMERGREEN

    mixed = (
        valid
        & np.isfinite(eg_share)
        & np.isfinite(sg_share)
        & (eg_share < threshold)
        & (sg_share < threshold)
    )
    out[mixed] = MIXED

    return out


def publication_filename(site):
    return (
        f"{site}_SiDroForest_contextual_"
        f"hardlabel_nominal500m_t075.tif"
    )


def write_publication_map(
    path,
    array,
    source_profile,
    *,
    site,
    site_name,
    window_pixels
):
    profile = source_profile.copy()

    for key in [
        "blockxsize",
        "blockysize",
        "tiled",
        "interleave",
        "photometric"
    ]:
        profile.pop(key, None)

    profile.update(
        driver="GTiff",
        count=1,
        dtype="uint8",
        nodata=0,
        compress="deflate"
    )

    with rasterio.open(path, "w", **profile) as dst:
        dst.write(array.astype("uint8"), 1)

        dst.update_tags(
            product="SiDroForest contextual forest-composition map",
            site_code=site,
            site_name=site_name,
            contextual_method="hard-label spatial composition",
            source="previously developed SiDroForest RF hard labels",
            nominal_neighbourhood_m=str(ADOPTED_SCALE_M),
            window_pixels=f"{window_pixels}x{window_pixels}",
            dominance_threshold=str(ADOPTED_THRESHOLD),
            class_0="masked/non-forest",
            class_1="evergreen needleleaf forest",
            class_2="summergreen needleleaf forest",
            class_3="mixed forest",
            interpretation=(
                "Mixed forest indicates local spatial coexistence of "
                "evergreen and summergreen hard RF labels; it is not "
                "a direct RF class."
            )
        )


In [ ]:
# ============================================================
# 4. GENERATE THE FINAL PUBLICATION MAPS
# ============================================================

metadata_rows = []

for _, row in selected.sort_values("site").iterrows():
    site = str(row["site"])
    source_file = Path(str(row["hard_file"]))
    site_name = SITE_NAMES.get(site, site)

    print("\n" + "=" * 70)
    print(site, "-", site_name)
    print("Source:", source_file)

    with rasterio.open(source_file) as src:
        hard = src.read(1).astype("uint8")
        profile = src.profile.copy()
        crs = src.crs
        transform = src.transform
        bounds = src.bounds
        source_nodata = src.nodata

    raw_classes = sorted(np.unique(hard).tolist())

    unexpected = sorted(
        set(raw_classes) - {0, EVERGREEN, SUMMERGREEN}
    )
    if unexpected:
        raise ValueError(
            f"{site}: unexpected hard-RF class values: {unexpected}"
        )

    xres = abs(transform.a)
    yres = abs(transform.e)

    if not np.isclose(xres, yres, rtol=0, atol=1e-6):
        raise ValueError(
            f"{site}: non-square source pixels: {xres} x {yres}"
        )

    window_pixels = odd_window_pixels(
        ADOPTED_SCALE_M,
        xres
    )

    valid = np.isin(
        hard,
        [EVERGREEN, SUMMERGREEN]
    )

    eg_share, sg_share = hard_context(
        hard,
        valid,
        window_pixels
    )

    contextual = categorical_from_composition(
        eg_share,
        sg_share,
        valid,
        ADOPTED_THRESHOLD
    )

    out_file = MAP_DIR / publication_filename(site)

    write_publication_map(
        out_file,
        contextual,
        profile,
        site=site,
        site_name=site_name,
        window_pixels=window_pixels
    )

    forest = np.isin(
        contextual,
        [EVERGREEN, SUMMERGREEN, MIXED]
    )
    forest_pixels = int(forest.sum())

    pixel_area_m2 = xres * yres
    pixel_area_km2 = pixel_area_m2 / 1_000_000.0

    counts = {
        cls: int(np.sum(contextual == cls))
        for cls in [0, 1, 2, 3]
    }

    def pct(cls):
        return (
            counts[cls] / forest_pixels * 100
            if forest_pixels else np.nan
        )

    metadata_rows.append({
        "site_code": site,
        "site_name": site_name,
        "filename": out_file.name,
        "source_hard_label_file": source_file.name,
        "contextual_method": "hard-label spatial composition",
        "nominal_neighbourhood_m": ADOPTED_SCALE_M,
        "window_pixels": window_pixels,
        "effective_window_width_m": window_pixels * xres,
        "dominance_threshold": ADOPTED_THRESHOLD,
        "crs": str(crs),
        "pixel_size_x": xres,
        "pixel_size_y": yres,
        "width_pixels": contextual.shape[1],
        "height_pixels": contextual.shape[0],
        "bounds_left": bounds.left,
        "bounds_bottom": bounds.bottom,
        "bounds_right": bounds.right,
        "bounds_top": bounds.top,
        "nodata_value": 0,
        "masked_or_nonforest_pixels": counts[0],
        "evergreen_pixels": counts[1],
        "summergreen_pixels": counts[2],
        "mixed_pixels": counts[3],
        "forest_pixels": forest_pixels,
        "evergreen_pct_of_forest": pct(1),
        "summergreen_pct_of_forest": pct(2),
        "mixed_pct_of_forest": pct(3),
        "mapped_forest_area_km2": forest_pixels * pixel_area_km2,
    })

    print("Saved:", out_file)
    print(
        "Classes (% of mapped forest):",
        f"evergreen={pct(1):.2f},",
        f"summergreen={pct(2):.2f},",
        f"mixed={pct(3):.2f}"
    )
    print(
        "Nominal 500 m setting ->",
        f"{window_pixels} x {window_pixels} pixels",
        f"(~{window_pixels * xres:.1f} m raster footprint)"
    )

metadata_df = pd.DataFrame(metadata_rows)
display(metadata_df)


## 3. Quality control

The checks below are intended to catch the most important publication errors:

- wrong class coding;
- missing CRS;
- unexpected raster resolution;
- inconsistent dimensions;
- accidental empty maps;
- divergence from already generated v16 contextual maps for Lake Khamra/Yakutsk.

For LK and YA, if the corresponding v16 map exists, the staged publication map must be **pixel-identical** to that final analysis output.


In [ ]:
# ============================================================
# 5. QUALITY CONTROL
# ============================================================

qc_rows = []

for _, meta in metadata_df.iterrows():
    site = meta["site_code"]
    path = MAP_DIR / meta["filename"]

    with rasterio.open(path) as src:
        arr = src.read(1)
        classes = sorted(np.unique(arr).tolist())

        qc = {
            "site_code": site,
            "file_exists": path.exists(),
            "crs_present": src.crs is not None,
            "class_values_valid":
                set(classes).issubset({0, 1, 2, 3}),
            "contains_forest":
                bool(np.isin(arr, [1, 2, 3]).any()),
            "pixel_size_x": abs(src.transform.a),
            "pixel_size_y": abs(src.transform.e),
            "square_pixels":
                bool(np.isclose(
                    abs(src.transform.a),
                    abs(src.transform.e),
                    atol=1e-6
                )),
            "classes_present": ",".join(
                map(str, classes)
            ),
        }

    # Compare against final v16/v17 map if available.
    existing = EXISTING_CONTEXT_DIR / (
        f"{site}_hard_label_"
        f"{ADOPTED_SCALE_M}m_"
        f"t{ADOPTED_THRESHOLD:.2f}.tif"
    )

    if existing.exists():
        with rasterio.open(existing) as src:
            old = src.read(1)

        with rasterio.open(path) as src:
            new = src.read(1)

        qc["existing_final_map_found"] = True
        qc["pixel_identical_to_existing_final"] = (
            old.shape == new.shape
            and np.array_equal(old, new)
        )
    else:
        qc["existing_final_map_found"] = False
        qc["pixel_identical_to_existing_final"] = np.nan

    qc_rows.append(qc)

qc_df = pd.DataFrame(qc_rows)
display(qc_df)

hard_fail_cols = [
    "file_exists",
    "crs_present",
    "class_values_valid",
    "contains_forest",
    "square_pixels",
]

if not qc_df[hard_fail_cols].all().all():
    raise RuntimeError(
        "One or more hard QC checks failed. "
        "Do not publish before resolving them."
    )

# Existing v16 maps, where present, must match.
existing_checks = qc_df[
    qc_df["existing_final_map_found"]
]

if len(existing_checks):
    if not existing_checks[
        "pixel_identical_to_existing_final"
    ].all():
        raise RuntimeError(
            "At least one staged map differs from an existing "
            "final v16/v17 contextual map."
        )

print("All hard QC checks passed.")


In [ ]:
# ============================================================
# 6. WRITE METADATA TABLES
# ============================================================

metadata_csv = METADATA_DIR / "contextual_map_metadata.csv"
metadata_df.to_csv(metadata_csv, index=False)

class_legend = pd.DataFrame([
    {
        "value": 0,
        "class_name": "masked / non-forest",
        "description": (
            "Pixels excluded by the SiDroForest forest mask; "
            "encoded as GeoTIFF nodata value 0."
        ),
    },
    {
        "value": 1,
        "class_name": "evergreen needleleaf forest",
        "description": (
            "Local evergreen hard-label proportion reaches "
            "the dominance threshold."
        ),
    },
    {
        "value": 2,
        "class_name": "summergreen needleleaf forest",
        "description": (
            "Local summergreen hard-label proportion reaches "
            "the dominance threshold."
        ),
    },
    {
        "value": 3,
        "class_name": "mixed forest",
        "description": (
            "Neither evergreen nor summergreen hard-label "
            "proportion reaches the dominance threshold."
        ),
    },
])

legend_csv = METADATA_DIR / "class_legend.csv"
class_legend.to_csv(legend_csv, index=False)

qc_csv = METADATA_DIR / "quality_control.csv"
qc_df.to_csv(qc_csv, index=False)

print("Saved:", metadata_csv)
print("Saved:", legend_csv)
print("Saved:", qc_csv)

display(class_legend)


In [ ]:
# ============================================================
# 7. GENERATE README
# ============================================================

DATASET_TITLE = (
    "Scale-aware contextual forest-composition maps derived "
    "from SiDroForest Sentinel-2 classifications at selected "
    "sites in Eastern Siberia"
)

readme = f"""# {DATASET_TITLE}

## Overview

This dataset contains final contextual forest-composition maps derived
from the previously developed SiDroForest Sentinel-2 Random Forest
forest-type classifications.

The maps distinguish:

- 1 = evergreen needleleaf forest
- 2 = summergreen needleleaf forest
- 3 = mixed forest

Pixels outside the retained forest mask are encoded as 0
(masked / non-forest; GeoTIFF nodata value = 0).

## Contextual forest-composition method

The source SiDroForest Random Forest provides hard pixel-level labels
for evergreen and summergreen needleleaf forest. The contextual maps
published here are derived from the local spatial composition of these
hard labels.

For each forest pixel, local evergreen and summergreen proportions are
calculated inside a centred neighbourhood. A pixel is assigned to
evergreen or summergreen forest when the corresponding local proportion
reaches the dominance threshold. If neither class reaches the threshold,
the pixel is assigned to mixed forest.

Final publication configuration:

- contextual method: hard-label spatial composition
- nominal neighbourhood width: {ADOPTED_SCALE_M} m
- dominance threshold: {ADOPTED_THRESHOLD:.2f}
- input/output raster spacing: inherited from the underlying
  SiDroForest classification (approximately 10 m for the published maps)

At 10 m pixel spacing, the centred nominal 500 m neighbourhood is
implemented as a 51 x 51 pixel window. Therefore, its raster footprint
is approximately 510 m across. The 500 m designation is retained as the
nominal analysis scale used in the manuscript.

The 500 m / 0.75 combination is an operational reference configuration,
not a statistically optimized ecological threshold. The associated
manuscript evaluates sensitivity across neighbourhood widths of
100, 250, 500, 750 and 1000 m and dominance thresholds from
0.50 to 0.90.

## Sites

The publication package currently contains the selected field-linked
manuscript sites:

{chr(10).join(f"- {s}: {SITE_NAMES.get(s, s)}" for s in metadata_df["site_code"].tolist())}

## Files

### maps/

GeoTIFF contextual forest-composition maps. Each GeoTIFF contains
embedded metadata tags describing the contextual method, neighbourhood
size, threshold and class definitions.

### metadata/contextual_map_metadata.csv

Per-map spatial metadata, class counts, class percentages and mapped
forest area.

### metadata/class_legend.csv

Class coding and definitions.

### metadata/quality_control.csv

Automated quality-control checks performed before packaging.

### SHA256SUMS.txt

SHA-256 checksums for all deposited files.

## Interpretation

The mixed-forest class is a contextual spatial-composition class. It
represents neighbourhoods in which confidently classified evergreen and
summergreen hard RF labels coexist without either class reaching the
0.75 dominance threshold. It should not be interpreted as a directly
learned Random Forest class or as a measure of RF classification
uncertainty.

The maps cover selected field-linked study sites and should not be
interpreted as a statistically representative sample of the complete
Siberian summergreen-evergreen forest transition zone.

## Relationship to existing SiDroForest datasets

The underlying Sentinel-2 imagery, field reference information,
classifier-development data and earlier SiDroForest map products are
described and/or published separately. This dataset specifically
contains the contextual forest-composition products introduced in the
revised study.

Existing related dataset cited in the manuscript:

van Geffen et al., High-resolution Sentinel-2 derived forest class maps
and aggregated upscaled forest class maps from the Copernicus Global
Land Service in selected regions of Western Yakutia and Chukotka,
Eastern Siberia, Russia. PANGAEA.
https://doi.org/10.1594/PANGAEA.973410

## Citation

[ADD THE FINAL DOI AND DATASET CITATION AFTER REPOSITORY PUBLICATION]

## License

[ADD THE LICENSE SELECTED FOR THE REPOSITORY, e.g. CC BY 4.0]

## Contact

Femke van Geffen
"""

readme_path = DATASET_ROOT / "README.md"
readme_path.write_text(readme, encoding="utf-8")

print(readme_path.read_text(encoding="utf-8")[:4000])
print("\nSaved:", readme_path)


In [ ]:
# ============================================================
# 8. SHA-256 CHECKSUM MANIFEST
# ============================================================

def sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


files_for_manifest = sorted(
    p for p in DATASET_ROOT.rglob("*")
    if p.is_file()
    and p.name not in {
        "SHA256SUMS.txt",
    }
    and not p.suffix.lower() == ".zip"
)

manifest_lines = []

for p in files_for_manifest:
    rel = p.relative_to(DATASET_ROOT)
    manifest_lines.append(
        f"{sha256(p)}  {rel.as_posix()}"
    )

manifest_path = DATASET_ROOT / "SHA256SUMS.txt"
manifest_path.write_text(
    "\n".join(manifest_lines) + "\n",
    encoding="utf-8"
)

print(manifest_path.read_text())


In [ ]:
# ============================================================
# 9. FINAL PACKAGE AUDIT
# ============================================================

print("DATASET PACKAGE")
print("=" * 80)

for p in sorted(DATASET_ROOT.rglob("*")):
    if p.is_file():
        print(
            f"{p.relative_to(DATASET_ROOT)}"
            f"  ({p.stat().st_size / 1024 / 1024:.2f} MB)"
        )

print("\nMap count:", len(list(MAP_DIR.glob("*.tif"))))
print("Expected selected sites:", len(selected))
print("Total mapped forest area (km²):",
      round(metadata_df["mapped_forest_area_km2"].sum(), 2))

print("\nBefore publication:")
print("1. Review the site list.")
print("2. Review metadata/contextual_map_metadata.csv.")
print("3. Open several GeoTIFFs in QGIS and visually inspect them.")
print("4. Add the repository licence.")
print("5. Upload the maps + metadata + README + SHA256SUMS.")
print("6. Add the assigned DOI to README.md and the manuscript.")


In [ ]:
# ============================================================
# 10. OPTIONAL — CREATE A ZIP ARCHIVE FOR UPLOAD
# ============================================================

zip_path = DATASET_ROOT.parent / (
    "SiDroForest_contextual_maps_publication_v1.zip"
)

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    for p in sorted(DATASET_ROOT.rglob("*")):
        if p.is_file():
            zf.write(
                p,
                arcname=Path(DATASET_ROOT.name)
                / p.relative_to(DATASET_ROOT)
            )

print("Created:", zip_path)
print(
    "ZIP size:",
    round(zip_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)
